# 01 · Kaggle validation and EDA

Run notebook 02 first. This performs the real image audit and creates plots plus the private duplicate-review files. It can take considerable time for 30,034 images.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/mamta-kumari28/Chest-X-Ray-Diagnosis-using-Deep-Learning-.git"
REPO_REF = "phase2/final-validation-fixes"  # Change to main after the PR is merged.
ROOT = Path("/kaggle/working/covid-xray-project")
OUTPUT_ROOT = Path("/kaggle/working/covid_xray_outputs")

if not ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, "--depth", "1", REPO_URL, str(ROOT)], check=True)
os.chdir(ROOT)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Repository:", ROOT)
print("Outputs:", OUTPUT_ROOT)

In [ ]:
# The attached Kaggle dataset must contain train/ and test/ image folders.
def find_data_root():
    candidates = [Path("/kaggle/input/covidx-cxr2")]
    candidates += sorted(Path("/kaggle/input").glob("**/train"))
    for item in candidates:
        root = item.parent if item.name == "train" else item
        if (root / "train").is_dir() and (root / "test").is_dir():
            return root
    raise FileNotFoundError("Attach Kaggle dataset andyczhao/covidx-cxr2 version 7 first.")

DATA_DIR = find_data_root()
print("Dataset:", DATA_DIR)

In [ ]:
def find_one(name, roots=(Path("/kaggle/working"), Path("/kaggle/input"))):
    matches = []
    for root in roots:
        matches.extend(root.glob(f"**/{name}"))
    matches = [path for path in matches if ".git" not in path.parts]
    if not matches:
        raise FileNotFoundError(f"Could not find {name}. Attach the previous notebook output as a Kaggle input.")
    print(name, "->", matches[0])
    return matches[0]

METADATA = find_one("patients.csv")

In [ ]:
from covid_xray.data import audit_metadata_file
from covid_xray.eda import create_dataset_report
from covid_xray.review import write_duplicate_review
import json
import pandas as pd

PROCESSED = OUTPUT_ROOT / "processed"
EDA = OUTPUT_ROOT / "eda"
PRIVATE = EDA / "private"
PROCESSED.mkdir(parents=True, exist_ok=True)
PRIVATE.mkdir(parents=True, exist_ok=True)

VALIDATED = PROCESSED / "validated_metadata.csv"
REPORT = EDA / "data_quality_report.json"
audited, report = audit_metadata_file(METADATA, DATA_DIR, VALIDATED, REPORT)
create_dataset_report(audited, report, EDA)
write_duplicate_review(
    VALIDATED,
    PRIVATE / "duplicate_review.csv",
    summary_path=EDA / "duplicate_review_summary.json",
    contact_sheets=PRIVATE / "contact_sheets",
    data_dir=DATA_DIR,
)
print(json.dumps(report, indent=2))

In [ ]:
from IPython.display import Image as DisplayImage, display

for name in ("class_distribution.png", "sample_images.png", "image_dimensions.png"):
    path = EDA / name
    if path.exists():
        display(DisplayImage(filename=str(path)))
print("Private review:", PRIVATE / "duplicate_review.csv")
print("Save the full covid_xray_outputs folder as this notebook's output.")

## Required human review before training

Download `duplicate_review.csv`. For every `perceptual:*` group, inspect the contact sheet and set one consistent `review_status` (`accepted` or `rejected`) plus a note. Keep patient IDs and paths private. Upload the completed file to a **private Kaggle dataset** named `duplicate_decisions.csv`. Do not mark candidates automatically.